In [1]:
import os
import numpy as np
import cv2
import shutil

# Set up YOLOv3 network and classes
net = cv2.dnn.readNet('C:\\Users\\uif68352\\Desktop\\ActiveLearning\\Yolo\\yolov3.weights', 'C:\\Users\\uif68352\\Desktop\\ActiveLearning\\Yolo\\Yolo config file.txt')
classes = []
with open('C:\\Users\\uif68352\\Desktop\\ActiveLearning\\Yolo\\COCO.txt') as f:
    classes = f.read().splitlines()
print(classes)

# Set up image folder and list to store uncertainty scores
folder_path = 'C:\\Users\\uif68352\\Desktop\\ActiveLearning\\Checking_YOLOv3\\YOLOv3\\Unlabelled images'
uncertainty_scores = []




['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [20]:
# Loop over all images in folder
for filename in os.listdir(folder_path):
    if filename.endswith('.jpeg') or filename.endswith('.png'):
        # Read image and get its dimensions
        img_path = os.path.join(folder_path, filename)
        img = cv2.imread(img_path)
        height, width, _ = img.shape

        # Preprocess image for YOLOv3
        blob = cv2.dnn.blobFromImage(img, 1/255, (416,416), (0,0,0), swapRB=True, crop=False)

        # Set input for YOLOv3 network and get output
        net.setInput(blob)
        output_layers_names = net.getUnconnectedOutLayersNames()
        layeroutputs = net.forward(output_layers_names)

        # Initialize lists to store detected objects' attributes
        boxes = []
        confidences = []
        class_ids = []

        # Loop over output from YOLOv3 network and extract detected objects' attributes
        for output in layeroutputs:
            for detection in output:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.5:
                    center_x = int(detection[0]*width)
                    center_y = int(detection[1]*height)
                    w = int(detection[2]*width)
                    h = int(detection[3]*height)

                    x = int(center_x - w/2)
                    y = int(center_y - h/2)

                    boxes.append([x, y ,w, h])
                    confidences.append(float(confidence))
                    print("confidences: ",confidences)
                    class_ids.append(class_id)
                    print("class_ids: ",class_ids)

        # Apply non-max suppression to remove overlapping boxes
        indexes = cv2.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)

        # Calculate uncertainty score for this image
        uncertainty_score = np.mean(sorted(confidences)[:5])
        uncertainty_scores.append((img_path, uncertainty_score))

# Sort uncertainty scores in ascending order and select top 10 most uncertain images
uncertainty_scores = sorted(uncertainty_scores, key=lambda x: x[1])
top_10_uncertain_images = [img_path for img_path, score in uncertainty_scores[:30]]

print('top_10_uncertain_images:',top_10_uncertain_images)



confidences:  [0.978560209274292]
class_ids:  [2]
confidences:  [0.978560209274292, 0.987713634967804]
class_ids:  [2, 2]
confidences:  [0.978560209274292, 0.987713634967804, 0.9869030117988586]
class_ids:  [2, 2, 2]
confidences:  [0.978560209274292, 0.987713634967804, 0.9869030117988586, 0.9866498112678528]
class_ids:  [2, 2, 2, 2]
confidences:  [0.978560209274292, 0.987713634967804, 0.9869030117988586, 0.9866498112678528, 0.5728963613510132]
class_ids:  [2, 2, 2, 2, 0]
confidences:  [0.978560209274292, 0.987713634967804, 0.9869030117988586, 0.9866498112678528, 0.5728963613510132, 0.5490592122077942]
class_ids:  [2, 2, 2, 2, 0, 9]
confidences:  [0.978560209274292, 0.987713634967804, 0.9869030117988586, 0.9866498112678528, 0.5728963613510132, 0.5490592122077942, 0.7919557690620422]
class_ids:  [2, 2, 2, 2, 0, 9, 9]
confidences:  [0.978560209274292, 0.987713634967804, 0.9869030117988586, 0.9866498112678528, 0.5728963613510132, 0.5490592122077942, 0.7919557690620422, 0.7930958867073059]


confidences:  [0.9934118390083313]
class_ids:  [2]
confidences:  [0.9934118390083313, 0.9942178130149841]
class_ids:  [2, 2]
confidences:  [0.9934118390083313, 0.9942178130149841, 0.6422158479690552]
class_ids:  [2, 2, 0]
confidences:  [0.9934118390083313, 0.9942178130149841, 0.6422158479690552, 0.8286677598953247]
class_ids:  [2, 2, 0, 9]
confidences:  [0.9934118390083313, 0.9942178130149841, 0.6422158479690552, 0.8286677598953247, 0.6614208817481995]
class_ids:  [2, 2, 0, 9, 9]
confidences:  [0.9934118390083313, 0.9942178130149841, 0.6422158479690552, 0.8286677598953247, 0.6614208817481995, 0.5416567921638489]
class_ids:  [2, 2, 0, 9, 9, 9]
confidences:  [0.9934118390083313, 0.9942178130149841, 0.6422158479690552, 0.8286677598953247, 0.6614208817481995, 0.5416567921638489, 0.6747538447380066]
class_ids:  [2, 2, 0, 9, 9, 9, 9]
confidences:  [0.9934118390083313, 0.9942178130149841, 0.6422158479690552, 0.8286677598953247, 0.6614208817481995, 0.5416567921638489, 0.6747538447380066, 0.893

confidences:  [0.68008953332901]
class_ids:  [0]
confidences:  [0.68008953332901, 0.9798421859741211]
class_ids:  [0, 2]
confidences:  [0.68008953332901, 0.9798421859741211, 0.9934243559837341]
class_ids:  [0, 2, 2]
confidences:  [0.68008953332901, 0.9798421859741211, 0.9934243559837341, 0.9906141757965088]
class_ids:  [0, 2, 2, 2]
confidences:  [0.68008953332901, 0.9798421859741211, 0.9934243559837341, 0.9906141757965088, 0.9923409223556519]
class_ids:  [0, 2, 2, 2, 2]
confidences:  [0.68008953332901, 0.9798421859741211, 0.9934243559837341, 0.9906141757965088, 0.9923409223556519, 0.6103578209877014]
class_ids:  [0, 2, 2, 2, 2, 0]
confidences:  [0.68008953332901, 0.9798421859741211, 0.9934243559837341, 0.9906141757965088, 0.9923409223556519, 0.6103578209877014, 0.7832598090171814]
class_ids:  [0, 2, 2, 2, 2, 0, 9]
confidences:  [0.68008953332901, 0.9798421859741211, 0.9934243559837341, 0.9906141757965088, 0.9923409223556519, 0.6103578209877014, 0.7832598090171814, 0.6448863744735718]
c

confidences:  [0.5069671273231506]
class_ids:  [0]
confidences:  [0.5069671273231506, 0.981406569480896]
class_ids:  [0, 2]
confidences:  [0.5069671273231506, 0.981406569480896, 0.990240752696991]
class_ids:  [0, 2, 2]
confidences:  [0.5069671273231506, 0.981406569480896, 0.990240752696991, 0.5675137639045715]
class_ids:  [0, 2, 2, 0]
confidences:  [0.5069671273231506, 0.981406569480896, 0.990240752696991, 0.5675137639045715, 0.7490290999412537]
class_ids:  [0, 2, 2, 0, 9]
confidences:  [0.5069671273231506, 0.981406569480896, 0.990240752696991, 0.5675137639045715, 0.7490290999412537, 0.5975587964057922]
class_ids:  [0, 2, 2, 0, 9, 9]
confidences:  [0.5069671273231506, 0.981406569480896, 0.990240752696991, 0.5675137639045715, 0.7490290999412537, 0.5975587964057922, 0.6690904498100281]
class_ids:  [0, 2, 2, 0, 9, 9, 9]
confidences:  [0.5069671273231506, 0.981406569480896, 0.990240752696991, 0.5675137639045715, 0.7490290999412537, 0.5975587964057922, 0.6690904498100281, 0.7467778921127319

confidences:  [0.6634224057197571]
class_ids:  [0]
confidences:  [0.6634224057197571, 0.9325743913650513]
class_ids:  [0, 2]
confidences:  [0.6634224057197571, 0.9325743913650513, 0.9594019651412964]
class_ids:  [0, 2, 2]
confidences:  [0.6634224057197571, 0.9325743913650513, 0.9594019651412964, 0.5622208118438721]
class_ids:  [0, 2, 2, 2]
confidences:  [0.6634224057197571, 0.9325743913650513, 0.9594019651412964, 0.5622208118438721, 0.8077210187911987]
class_ids:  [0, 2, 2, 2, 2]
confidences:  [0.6634224057197571, 0.9325743913650513, 0.9594019651412964, 0.5622208118438721, 0.8077210187911987, 0.863701581954956]
class_ids:  [0, 2, 2, 2, 2, 0]
confidences:  [0.6634224057197571, 0.9325743913650513, 0.9594019651412964, 0.5622208118438721, 0.8077210187911987, 0.863701581954956, 0.6312050223350525]
class_ids:  [0, 2, 2, 2, 2, 0, 9]
confidences:  [0.6634224057197571, 0.9325743913650513, 0.9594019651412964, 0.5622208118438721, 0.8077210187911987, 0.863701581954956, 0.6312050223350525, 0.770592

...

confidences:  [0.5332834124565125]
class_ids:  [2]
confidences:  [0.5332834124565125, 0.9028921723365784]
class_ids:  [2, 2]
confidences:  [0.5332834124565125, 0.9028921723365784, 0.882086455821991]
class_ids:  [2, 2, 2]
confidences:  [0.5332834124565125, 0.9028921723365784, 0.882086455821991, 0.8563401699066162]
class_ids:  [2, 2, 2, 2]
confidences:  [0.5332834124565125, 0.9028921723365784, 0.882086455821991, 0.8563401699066162, 0.9551687836647034]
class_ids:  [2, 2, 2, 2, 2]
confidences:  [0.5332834124565125, 0.9028921723365784, 0.882086455821991, 0.8563401699066162, 0.9551687836647034, 0.5369727611541748]
class_ids:  [2, 2, 2, 2, 2, 2]
confidences:  [0.5332834124565125, 0.9028921723365784, 0.882086455821991, 0.8563401699066162, 0.9551687836647034, 0.5369727611541748, 0.9499762654304504]
class_ids:  [2, 2, 2, 2, 2, 2, 2]
confidences:  [0.9617790579795837]
class_ids:  [2]
confidences:  [0.9617790579795837, 0.7447590231895447]
class_ids:  [2, 2]
confidences:  [0.9617790579795837, 0.744

confidences:  [0.9941514134407043]
class_ids:  [2]
confidences:  [0.9941514134407043, 0.9629241824150085]
class_ids:  [2, 2]
confidences:  [0.9941514134407043, 0.9629241824150085, 0.5059307217597961]
class_ids:  [2, 2, 0]
confidences:  [0.9941514134407043, 0.9629241824150085, 0.5059307217597961, 0.8321686387062073]
class_ids:  [2, 2, 0, 2]
confidences:  [0.9941514134407043, 0.9629241824150085, 0.5059307217597961, 0.8321686387062073, 0.6736798286437988]
class_ids:  [2, 2, 0, 2, 0]
confidences:  [0.9941514134407043, 0.9629241824150085, 0.5059307217597961, 0.8321686387062073, 0.6736798286437988, 0.5643906593322754]
class_ids:  [2, 2, 0, 2, 0, 0]
confidences:  [0.9941514134407043, 0.9629241824150085, 0.5059307217597961, 0.8321686387062073, 0.6736798286437988, 0.5643906593322754, 0.726202130317688]
class_ids:  [2, 2, 0, 2, 0, 0, 2]
confidences:  [0.9941514134407043, 0.9629241824150085, 0.5059307217597961, 0.8321686387062073, 0.6736798286437988, 0.5643906593322754, 0.726202130317688, 0.95206

confidences:  [0.976342499256134]
class_ids:  [2]
confidences:  [0.976342499256134, 0.9562975764274597]
class_ids:  [2, 2]
confidences:  [0.976342499256134, 0.9562975764274597, 0.9800772666931152]
class_ids:  [2, 2, 2]
confidences:  [0.976342499256134, 0.9562975764274597, 0.9800772666931152, 0.9754331707954407]
class_ids:  [2, 2, 2, 2]
confidences:  [0.976342499256134, 0.9562975764274597, 0.9800772666931152, 0.9754331707954407, 0.5907289385795593]
class_ids:  [2, 2, 2, 2, 0]
confidences:  [0.976342499256134, 0.9562975764274597, 0.9800772666931152, 0.9754331707954407, 0.5907289385795593, 0.6384053826332092]
class_ids:  [2, 2, 2, 2, 0, 0]
confidences:  [0.976342499256134, 0.9562975764274597, 0.9800772666931152, 0.9754331707954407, 0.5907289385795593, 0.6384053826332092, 0.8300006985664368]
class_ids:  [2, 2, 2, 2, 0, 0, 2]
confidences:  [0.976342499256134, 0.9562975764274597, 0.9800772666931152, 0.9754331707954407, 0.5907289385795593, 0.6384053826332092, 0.8300006985664368, 0.77611911296

confidences:  [0.8512637615203857]
class_ids:  [2]
confidences:  [0.8512637615203857, 0.9924696087837219]
class_ids:  [2, 2]
confidences:  [0.8512637615203857, 0.9924696087837219, 0.6972783803939819]
class_ids:  [2, 2, 2]
confidences:  [0.8512637615203857, 0.9924696087837219, 0.6972783803939819, 0.6899714469909668]
class_ids:  [2, 2, 2, 2]
confidences:  [0.8512637615203857, 0.9924696087837219, 0.6972783803939819, 0.6899714469909668, 0.550165593624115]
class_ids:  [2, 2, 2, 2, 2]
confidences:  [0.8512637615203857, 0.9924696087837219, 0.6972783803939819, 0.6899714469909668, 0.550165593624115, 0.6258297562599182]
class_ids:  [2, 2, 2, 2, 2, 2]
confidences:  [0.8512637615203857, 0.9924696087837219, 0.6972783803939819, 0.6899714469909668, 0.550165593624115, 0.6258297562599182, 0.9851418137550354]
class_ids:  [2, 2, 2, 2, 2, 2, 2]
confidences:  [0.8512637615203857, 0.9924696087837219, 0.6972783803939819, 0.6899714469909668, 0.550165593624115, 0.6258297562599182, 0.9851418137550354, 0.9831225

confidences:  [0.9832513928413391]
class_ids:  [0]
confidences:  [0.9832513928413391, 0.7172500491142273]
class_ids:  [0, 0]
confidences:  [0.9832513928413391, 0.7172500491142273, 0.8472675085067749]
class_ids:  [0, 0, 0]
confidences:  [0.9832513928413391, 0.7172500491142273, 0.8472675085067749, 0.7824286222457886]
class_ids:  [0, 0, 0, 0]
confidences:  [0.9748093485832214]
class_ids:  [0]
confidences:  [0.9748093485832214, 0.7249900698661804]
class_ids:  [0, 0]
confidences:  [0.9748093485832214, 0.7249900698661804, 0.5690007209777832]
class_ids:  [0, 0, 0]
confidences:  [0.9748093485832214, 0.7249900698661804, 0.5690007209777832, 0.8802813291549683]
class_ids:  [0, 0, 0, 0]
confidences:  [0.902558445930481]
class_ids:  [0]
confidences:  [0.902558445930481, 0.9437509179115295]
class_ids:  [0, 0]
confidences:  [0.902558445930481, 0.9437509179115295, 0.8113940358161926]
class_ids:  [0, 0, 0]
confidences:  [0.902558445930481, 0.9437509179115295, 0.8113940358161926, 0.6906924247741699]
cla

In [21]:
# Move top 10 most uncertain images to another folder
for img_path in top_10_uncertain_images:
    filename = os.path.basename(img_path)
    dst_path = 'C:\\Users\\uif68352\\Desktop\\ActiveLearning\\Checking_YOLOv3\\YOLOv3\\Labelled_images_new2\\' + filename
    shutil.move(img_path, dst_path)

print('Top 30 most uncertain images moved to another folder.')


Top 30 most uncertain images moved to another folder.
